# Structured Output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = ChatGroq(model="llama-3.1-8b-instant", temperature = 0.7)


In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str = Field(description="Movies actor name")
    role:str = Field(description="Role of actor in movie")

class Movie(BaseModel):
    title:str = Field(description="Name of the movie")
    year:int = Field(description="Movie released year")
    rating:float = Field(description="Movie rating", default= 0)
    cast:list[Actor] = Field(description="list of actors in movie" )


In [12]:
structured_movie_model = model.with_structured_output(Movie)
response = structured_movie_model.invoke("Give me detailes about 3 idiots movie")
response

Movie(title='3 Idiots', year=2009, rating=8.6, cast=[Actor(name='Aamir Khan', role='Rancho'), Actor(name='Kareena Kapoor', role='Pia Sahastrabuddhe'), Actor(name='R. Madhavan', role='Farhan Qureshi')])

## Typed Dict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [13]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

# DataClass
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorato

In [8]:
from dataclasses import  dataclass
from langchain.agents import  create_agent
from langchain.messages import  HumanMessage

@dataclass
class User:
    """Contact information of user"""
    name:str #User name
    email:str #User email
    address:str #User address

agent = create_agent("groq:llama-3.1-8b-instant", name="test-structured-op", response_format=User)
response = agent.invoke({
    "messages": [ HumanMessage("Extract info from : Hi my name is Prasad Bhosale. you can contact me on pr@gmail.com, I'm from Pune.")]
})
response["messages"][-1].content

"Returning structured response: User(name='Prasad Bhosale', email='pr@gmail.com', address='Pune')"

In [13]:
from dataclasses import  dataclass
from langchain_groq import ChatGroq
from langchain.messages import  HumanMessage

@dataclass
class User:
    """Contact information of user"""
    name:str #User name
    email:str #User email
    address:str #User address

model = ChatGroq(model="llama-3.1-8b-instant", name="test-structured-op")
structured_model = model.with_structured_output(User)
response = structured_model.invoke(
  "Extract info from : Hi my name is Prasad Bhosale. you can contact me on pr@gmail.com, I'm from Pune."
)
response

{'address': 'Pune', 'email': 'pr@gmail.com', 'name': 'Prasad Bhosale'}